# ML System Design: The Framework

Every ML system design interview answer should follow the same 7-step structure regardless of the problem. This note documents the framework, what to say at each step, and how to adapt it to the four major archetypes: recsys, search/ads, trust & safety, and LLM applications.

## What Interviewers Test
- Can you scope ambiguous requirements before diving in?
- Do you define metrics before choosing a model?
- Do you start with a simple baseline rather than jumping to complex models?
- Can you reason about the serving architecture and latency budget?
- Do you proactively address monitoring and failure modes?

## The 7-Step Framework

| Step | Time budget (45-min round) | What to produce |
|---|---|---|
| 1. Clarify requirements | 3–5 min | Problem scope, scale, constraints |
| 2. Define metrics | 3–5 min | Offline + online metrics |
| 3. Data & labeling | 5–7 min | Sources, labeling strategy, pitfalls |
| 4. Feature engineering | 5–7 min | Feature families, freshness needs |
| 5. Modeling | 8–10 min | Baseline → improved model |
| 6. Serving architecture | 5–7 min | Request flow, latency budget |
| 7. Monitoring & iteration | 3–5 min | Drift, alerts, feedback loop |


## Step 1: Clarify Requirements & Scale

**What to ask:**
- What is the business objective? (engagement? revenue? safety?)
- Who are the users? How many? (DAU, QPS)
- What is the acceptable latency? (p99 SLA)
- Is this a new product or improving an existing one?
- Are there hard constraints? (regulatory, cold-start, budget)

**Red flags:** Jumping straight to model choice. Not asking about scale. Not asking what "success" means to the PM.

> 💡 **Interview Tip:** Spend 3–5 minutes here. Interviewers specifically check whether you scope the problem before solving it. Say: *"Before diving in, let me clarify a few things…"*


## Step 2: Define Metrics

Always define BOTH offline and online metrics.

| Metric type | Description | Examples |
|---|---|---|
| **North-star (online)** | Business metric you ultimately optimize | DAU, revenue, watch-time |
| **Proxy (offline)** | What you can measure without deployment | AUC, NDCG, log-loss |
| **Guardrail (online)** | What must NOT get worse | Latency p99, abuse rate, churn |
| **Counter (online)** | Leading indicators of downstream harm | Click-bait rate, diversity |

**The proxy-metric pitfall:** Optimizing click-rate can degrade long-term engagement if it causes clickbait. Define guardrails upfront.


In [ ]:
import numpy as np

# --- Illustrative: NDCG computation (a key offline metric for ranking) ---
def dcg(relevances, k=None):
    """Discounted Cumulative Gain."""
    r = np.array(relevances[:k], dtype=float)
    if len(r) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(r)+2))
    return (r / discounts).sum()

def ndcg(relevances, k=None):
    """Normalized DCG: DCG / IDCG."""
    ideal = dcg(sorted(relevances, reverse=True), k=k)
    if ideal == 0:
        return 0.0
    return dcg(relevances, k=k) / ideal

# Example: two ranking systems on same query
system_a = [3, 2, 1, 0, 0]   # good ordering
system_b = [0, 0, 1, 2, 3]   # reverse ordering

print(f"System A NDCG@5: {ndcg(system_a, k=5):.4f}")
print(f"System B NDCG@5: {ndcg(system_b, k=5):.4f}")
print(f"NDCG@3 A: {ndcg(system_a, k=3):.4f}, B: {ndcg(system_b, k=3):.4f}")


## Steps 3–4: Data & Features (Summary)

**Label collection strategies:**
- **Explicit:** user ratings, thumbs up/down — sparse but reliable
- **Implicit:** clicks, watch-time, shares — abundant but noisy (click ≠ satisfaction)
- **Human labeling:** high quality but expensive; use for high-stakes problems
- **Weak supervision:** heuristic labelers (Snorkel pattern) — fast but noisy

**Feature families (recsys/ads):**
- **User features:** demographics, history, session context
- **Item features:** content embeddings, metadata, engagement stats
- **Context features:** time of day, device, location
- **Cross features:** user × item interaction history, co-occurrence

**Feature freshness:** Some features need near-real-time updates (session context), others can be precomputed daily (embeddings). This drives feature store design.


## Step 5: Modeling (Baseline → v2)

**The baseline doctrine:** Always start with a non-ML baseline.

| Stage | Model | Why |
|---|---|---|
| Baseline | Popularity ranker or rule-based | Zero training cost; sets the floor |
| v1 | Logistic regression on handcrafted features | Interpretable; fast to iterate |
| v2 | Gradient boosted trees (GBDT) or two-tower neural | Better generalization |
| v3 | Multi-task learning / large-scale DNN | Optimize multiple objectives |

**Two-tower pattern:** Common for retrieval.
- Query tower: encodes user/query context → query embedding
- Item tower: encodes item features → item embedding
- Score: dot product at query time


## Step 6: Serving Architecture

**The retrieval funnel (numbers matter):**

```
All items (~10M)
  ↓ [ANN retrieval / two-tower]   < 10ms
Candidates (~1,000)
  ↓ [Light ranker / GBDT]        < 20ms
Top-N (~100)
  ↓ [Full DNN ranker]            < 50ms
Re-ranked (~10)
  ↓ [Business rules / diversity]
Final result
```

> 💡 **Interview Tip:** Mention specific latency numbers (10ms, 20ms, 50ms). Interviewers notice whether you treat latency as a real constraint or hand-wave it.


## Step 7: Monitoring & Iteration

| Layer | What to monitor | Alert trigger |
|---|---|---|
| System | QPS, latency p99, error rate | > 2× baseline |
| Data | Feature distribution (PSI) | PSI > 0.2 |
| Model | Prediction distribution | Mean shift > 2σ |
| Business | Online metric (CTR, watch-time) | Significant drop via A/B |
| Labels | Label rate, label delay | Delay > SLA |


## Archetype Cheat Sheet

| Archetype | Retrieval | Ranker | Key metric | Key pitfall |
|---|---|---|---|---|
| **Recsys (feed)** | Two-tower ANN | DNN + multi-task | Watch-time, DAU | Filter bubble, cold-start |
| **Search** | BM25 + semantic | LTR (pairwise/listwise) | NDCG, MRR | Query understanding, freshness |
| **Ads (CTR)** | Targeting + retrieval | CTR model (logistic/DNN) | Revenue, CTR | Calibration for auction, position bias |
| **Trust & Safety** | Rule triggers | Classifier | Precision@review | Label delay, adversarial adaptation |
| **LLM app** | RAG retrieval | Re-ranker | Task completion, CSAT | Hallucination, latency/cost |


## Common Interview Questions

**Q: How do you handle the cold-start problem?**
For new users: use demographic/context features, popular item fallback, or onboarding flow to collect initial preferences. For new items: use content-based features (embeddings from text/image), editor curation, or exploration policies. The two-tower model handles new items better than collaborative filtering because it doesn't require interaction history.

**Q: What's the difference between online and offline metrics?**
Offline metrics (AUC, NDCG) are computed on a held-out dataset before deployment — fast and cheap but imperfect proxies. Online metrics (CTR, revenue) are the true business signals measured via A/B test after deployment. A model can improve offline metrics while degrading online metrics (Goodhart's law / proxy-metric failure).

**Q: How do you decide when to retrain?**
Monitor prediction distribution drift (PSI on scores) and business metric trends. Trigger retraining when PSI > threshold (typically 0.2) or when online metrics degrade beyond a guardrail. Some systems retrain continuously (online learning) for fast-moving signals like news; others batch-retrain daily/weekly for stable user preferences.

**Q: How would you handle a 45-minute design interview if you can't cover everything?**
Prioritize breadth over depth. Cover all 7 steps briefly (1–2 minutes each) in the first 15 minutes, then go deep on the 2–3 steps the interviewer probes. Signal that you know what you're glossing over: "I'm simplifying the feature store — happy to go deeper if you'd like."

**Q: What makes an MLE system design answer different from a generic SWE system design?**
MLEs must define metrics before architecture, discuss training pipelines and data freshness, address model-specific failure modes (drift, distribution shift), and reason about the feedback loop between model outputs and future training data — none of which appear in SWE design rounds.

## Key Takeaways
- Use the 7-step framework every time: requirements → metrics → data → features → model → serving → monitoring
- Always define both offline and online metrics, and name at least one guardrail
- Start with a non-ML baseline; justify complexity upgrades explicitly
- The retrieval funnel: millions → thousands → hundreds → tens, with latency at each stage
- Cold-start and feedback loops are the two universal failure modes — address them proactively
- Time budget: ~5 min requirements, ~5 min metrics, ~10 min model, ~5 min serving, ~5 min monitoring